# 01 — Extract and Merge CR-FIQA Scores

This notebook performs the main data-preparation pipeline:

1. Run the shared Colab setup from `00_colab_setup.ipynb`.
2. Load the CR-FIQA model checkpoint.
3. calculate a CR-FIQA score for every image in `DiveFace_subset/`.
4. save the generated scores.
5. merge the scores with the DiveFace annotation table.
6. save the merged dataset for exploratory analysis and machine learning.

## Required files

The following files must be available in the project folder configured in
`00_colab_setup.ipynb`:

- `DiveFace_subset/`
- `DiveFace_subset_annotations.pkl`
- the CR-FIQA model checkpoint

Keep `00_colab_setup.ipynb` and this notebook in the same `notebooks/` folder.


In [ ]:
# Locate and run the shared setup notebook.
#
# Recommended repository layout:
# fiqa-demographic-analysis/
# └── notebooks/
#     ├── 00_colab_setup.ipynb
#     └── 01_extract_and_merge_cr_fiqa_scores.ipynb

from pathlib import Path

setup_candidates = [
    Path.cwd() / "00_colab_setup.ipynb",
    Path.cwd() / "notebooks" / "00_colab_setup.ipynb",
    Path.cwd().parent / "notebooks" / "00_colab_setup.ipynb",
    Path("/content/drive/MyDrive/FIQA_Project/notebooks/00_colab_setup.ipynb"),
    Path("/content/drive/MyDrive/FIQA_Project/00_colab_setup.ipynb"),
]

SETUP_NOTEBOOK = next(
    (path for path in setup_candidates if path.exists()),
    None,
)

if SETUP_NOTEBOOK is None:
    checked_paths = "\n".join(
        f"- {path}" for path in setup_candidates
    )
    raise FileNotFoundError(
        "00_colab_setup.ipynb could not be found.\n"
        "Keep both notebooks in the same notebooks/ folder or update "
        "setup_candidates.\n\n"
        f"Checked:\n{checked_paths}"
    )

print(f"Running setup notebook: {SETUP_NOTEBOOK}")

get_ipython().run_line_magic(
    "run",
    f'"{SETUP_NOTEBOOK}"',
)


In [ ]:
# Imports and output paths

import sys

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

SCORES_OUTPUT_FILE = PROJECT_PATH / "diveface_cr_fiqa_scores.csv"
MERGED_OUTPUT_FILE = PROJECT_PATH / "diveface_fiqa_merged.csv"

# These variables are defined by 00_colab_setup.ipynb:
required_setup_variables = {
    "PROJECT_PATH": PROJECT_PATH,
    "IMAGE_FOLDER": IMAGE_FOLDER,
    "ANNOTATION_FILE": ANNOTATION_FILE,
    "MODEL_PATH": MODEL_PATH,
    "CR_FIQA_PATH": CR_FIQA_PATH,
}

print("Shared setup variables loaded successfully.")
for name, value in required_setup_variables.items():
    print(f"{name}: {value}")


In [ ]:
# Import the CR-FIQA architecture

cr_fiqa_path = str(CR_FIQA_PATH)

if cr_fiqa_path not in sys.path:
    sys.path.insert(0, cr_fiqa_path)

from backbones.iresnet import iresnet100


In [ ]:
# Select the computation device and load the CR-FIQA checkpoint

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

model = iresnet100(
    num_features=512,
    qs=1,
    use_se=False,
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
)

# Support both a plain state dictionary and checkpoints that wrap it.
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    state_dict = checkpoint["state_dict"]
else:
    state_dict = checkpoint

# Remove a possible DataParallel prefix.
state_dict = {
    key.removeprefix("module."): value
    for key, value in state_dict.items()
}

model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

print("CR-FIQA model loaded successfully.")


In [ ]:
# Image preprocessing

def preprocess_image(image_path: Path) -> np.ndarray | None:
    """Load and preprocess one image for the CR-FIQA model."""

    image = cv2.imread(str(image_path))

    if image is None:
        return None

    image = cv2.resize(
        image,
        (112, 112),
        interpolation=cv2.INTER_LINEAR,
    )

    # CR-FIQA's reference evaluation pipeline uses OpenCV BGR ordering.
    image = image.transpose(2, 0, 1)
    return image.astype(np.float32)


In [ ]:
# Collect image paths

SUPPORTED_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
}

image_paths = sorted(
    path
    for path in IMAGE_FOLDER.rglob("*")
    if path.is_file()
    and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

print(f"Number of images found: {len(image_paths)}")

if not image_paths:
    raise RuntimeError(
        f"No supported images were found in: {IMAGE_FOLDER}"
    )


In [ ]:
# CR-FIQA score generation

def generate_fiqa_scores(
    image_paths: list[Path],
    model: torch.nn.Module,
    device: torch.device,
    batch_size: int = 32,
) -> tuple[pd.DataFrame, list[str]]:
    """Generate CR-FIQA scores for a collection of images."""

    results: list[dict] = []
    unreadable_images: list[str] = []

    with torch.inference_mode():
        for start_index in tqdm(
            range(0, len(image_paths), batch_size),
            desc="Generating CR-FIQA scores",
        ):
            batch_paths = image_paths[
                start_index:start_index + batch_size
            ]

            batch_images = []
            valid_paths = []

            for image_path in batch_paths:
                image = preprocess_image(image_path)

                if image is None:
                    unreadable_images.append(str(image_path))
                    continue

                batch_images.append(image)
                valid_paths.append(image_path)

            if not batch_images:
                continue

            batch_array = np.stack(batch_images)
            batch_tensor = torch.from_numpy(batch_array).to(device)

            # Normalize pixel values from [0, 255] to [-1, 1].
            batch_tensor = (
                batch_tensor
                .div(255.0)
                .sub(0.5)
                .div(0.5)
            )

            _, quality_scores = model(batch_tensor)

            quality_scores = (
                quality_scores
                .detach()
                .cpu()
                .numpy()
                .reshape(-1)
            )

            for image_path, score in zip(
                valid_paths,
                quality_scores,
            ):
                relative_path = image_path.relative_to(
                    IMAGE_FOLDER
                ).as_posix()

                results.append(
                    {
                        "index": relative_path,
                        "file_name": image_path.name,
                        "cr_fiqa_score": float(score),
                    }
                )

    return pd.DataFrame(results), unreadable_images


In [ ]:
# Generate and save CR-FIQA scores

BATCH_SIZE = 32

scores, unreadable_images = generate_fiqa_scores(
    image_paths=image_paths,
    model=model,
    device=device,
    batch_size=BATCH_SIZE,
)

if scores.empty:
    raise RuntimeError("No CR-FIQA scores were generated.")

if unreadable_images:
    print(
        f"Warning: {len(unreadable_images)} images could not be loaded."
    )

if scores["index"].duplicated().any():
    duplicate_count = int(scores["index"].duplicated().sum())
    raise ValueError(
        f"The generated score table contains {duplicate_count} "
        "duplicate image indices."
    )

scores.to_csv(
    SCORES_OUTPUT_FILE,
    index=False,
)

print(f"Generated scores: {len(scores)}")
print(f"Scores saved to: {SCORES_OUTPUT_FILE}")

display(scores.head())

print("\nScore statistics:")
display(scores["cr_fiqa_score"].describe())


In [ ]:
# Load and normalize the annotation table

annotations = pd.read_pickle(ANNOTATION_FILE).copy()

print(f"Annotation rows: {len(annotations)}")
print(f"Annotation columns: {len(annotations.columns)}")

if "index" not in annotations.columns:
    raise KeyError(
        "The annotation table does not contain an 'index' column."
    )

# Normalize path separators and remove an optional dataset-folder prefix.
annotations["index"] = (
    annotations["index"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
    .str.removeprefix("DiveFace_subset/")
    .str.removeprefix("DiveFace_subsampled/")
)

if annotations["index"].duplicated().any():
    duplicate_count = int(
        annotations["index"].duplicated().sum()
    )
    raise ValueError(
        f"The annotation table contains {duplicate_count} "
        "duplicate image indices."
    )

display(annotations.head())


In [ ]:
# Merge annotations and CR-FIQA scores

print(f"Annotations before merge: {annotations.shape}")
print(f"Scores before merge:      {scores.shape}")

merged = annotations.merge(
    scores[["index", "cr_fiqa_score"]],
    on="index",
    how="inner",
    validate="one_to_one",
)

matched_images = len(merged)
unmatched_annotations = len(annotations) - matched_images
unmatched_scores = len(scores) - matched_images

print(f"\nMatched images:         {matched_images}")
print(f"Unmatched annotations: {unmatched_annotations}")
print(f"Unmatched scores:      {unmatched_scores}")
print(f"Merged dataset shape:  {merged.shape}")

if merged.empty:
    raise RuntimeError(
        "The merge produced no rows. Check whether the relative image "
        "paths match the annotation index."
    )

match_rate = matched_images / len(scores)

if match_rate < 0.95:
    print(
        f"Warning: Only {match_rate:.1%} of generated scores were "
        "matched to annotation rows."
    )

display(merged.head())


In [ ]:
# Save the merged dataset

merged.to_csv(
    MERGED_OUTPUT_FILE,
    index=False,
)

print(f"Merged dataset saved to: {MERGED_OUTPUT_FILE}")
print("Score extraction and annotation merge completed successfully.")
